## Import

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import json
import time

## judge_model

In [2]:
judge_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

judge_tokenizer = AutoTokenizer.from_pretrained(
    judge_model_name
)

judge_model = AutoModelForCausalLM.from_pretrained(
    judge_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Judge model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Judge model loaded successfully!


In [3]:
def build_factuality_prompt(article, summary):

    prompt = f"""
You are an evaluator for Arabic text summarization.

Your task is to evaluate whether the generated summary is
FACTUALLY SUPPORTED by the source article.

Do NOT compare it with any reference summary.

SOURCE ARTICLE:
{article}

GENERATED SUMMARY:
{summary}

Evaluate the summary using these criteria:

1. Factuality:
   Are the claims in the summary supported by the article?

2. Relevance:
   Does the summary focus on information from the article?

3. Hallucination:
   Does the summary contain information that cannot be supported
   by the article?

Return ONLY valid JSON:

{{
    "factuality": 1-5,
    "relevance": 1-5,
    "hallucination": true/false,
    "explanation": "short explanation"
}}
"""

    return prompt

In [7]:
results_df = pd.read_csv(
    "../results/qwen2.5-1.5b-results.csv"
)

print(results_df.shape)
results_df.head()

(200, 5)


,id,text,reference_summary,prediction,inference_time
0,0,حلت أمس الإثنين عناصر من الفرقة الوطنية للشرطة...,اختلالات مشروع “كُورْنِيشْ آسفي”.. الفرقة الوط...,"عناصر من الشرطة القضائية في آسفي اتصلوا بـ""الف...",4.422769
1,1,علمت “كود” ان ندوة الرؤساء بمجلس المستشارين ال...,نايضة على بنشماس على لفلوس عاود ثاني . علاش كن...,"""كود"" أفاد بأن النقاش الساخن خلال ندوة الرؤساء...",5.759217
2,2,فالمغرب عارفين ومتأكدين باللي هاد القضية ديال ...,باش التقاليد كتغلب الجايحة. لمغاربة كيزورو وحد...,ر اكيد كاين فيهم اللي كيظل يدور وفهاد الدوران ...,5.746669
3,3,قالت مصادر محلية، أن لجنة تفتيش من وزارة الداخ...,لجنة تفتيشية من الداخلية طارت لجماعات فإقليم ا...,هناك لجنة تفتيش من وزارة الداخلية في المغرب، ت...,2.361186
4,4,قالت صفحة “اسرائيل تتكلم بالعربية”على “الفايسب...,اسرائيل لاعبة كبير وبالزربة مع المغرب. سفيرها ...,"قالت صفحة ""إسرائيل تتكلم بالعربية"" على ""فيسبوك...",14.969733


In [11]:
article = results_df.iloc[0]["text"]
summary = results_df.iloc[0]["prediction"]

prompt = build_factuality_prompt(
    article,
    summary
)
print(prompt)


You are an evaluator for Arabic text summarization.

Your task is to evaluate whether the generated summary is
FACTUALLY SUPPORTED by the source article.

Do NOT compare it with any reference summary.

SOURCE ARTICLE:
حلت أمس الإثنين عناصر من الفرقة الوطنية للشرطة القضائية بمدينة آسفي، قادة إليها من مدينة الدار البيضاء، وذلك لمباشرة الأبحاث التمهيدية الضرورية في شأن الشكاية التي أحيلت عليها من طرف الوكيل العام للملك بمراكش حول وجود شبهة اختلالات في مشروع إنجاز “كورنيش” آسفي. وحسب رئيس الجمعية المغربية لحماية المال العام، محمد الغلوسي، فإنه سبق للفرقة الوطنية للشرطة القضائية أن اتصلت في وقت سابق بصافي الدين البودالي بإعتباره رئيسا للفرع الجهوي مراكش الجنوب للجمعية قصد حضور المعاينة التي ستجرى من طرف ذات الفرقة على الأشغال المنجزة بكورنيش آسفي. وأشار إلى أن الفرع الجهوي مراكش الجنوب للجمعية المغربية لحماية المال العام سبق له أن تقدم في وقت سابق بشكاية الى الوكيل العام للملك لدى محكمة الإستئناف بمراكش حول ما اعتبره “اختلالات قانونية وتقنية ومالية”، والتي اعترت إنجاز مشروع كورنيش آسفي. وأ

In [12]:
messages = [
    {
        "role": "user",
        "content": prompt
    }
]

formatted_prompt = judge_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = judge_tokenizer(
    formatted_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=2048
).to(judge_model.device)

In [13]:
with torch.no_grad():

    outputs = judge_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

judge_output = judge_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(judge_output)

```json
{
    "factuality": 4,
    "relevance": 4,
    "hallucination": false,
    "explanation": "The summary accurately captures key points from the source article without adding or omitting significant details."
}
```


## judge_Function

In [15]:
def evaluate_factuality(article, summary):

    prompt = build_factuality_prompt(
        article,
        summary
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = judge_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = judge_tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(judge_model.device)

    with torch.no_grad():
        outputs = judge_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    output = judge_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return output

In [16]:
test_output = evaluate_factuality(
    results_df.iloc[0]["text"],
    results_df.iloc[0]["prediction"]
)

print(test_output)

```json
{
    "factuality": 4,
    "relevance": 4,
    "hallucination": false,
    "explanation": "The summary accurately captures key points from the source article without adding or omitting significant details."
}
```


## JSON Parser

In [22]:
def parse_judge_output(output):

    output = output.strip()

    # Remove markdown code fences if they exist
    if output.startswith("```"):
        output = output.replace("```json", "")
        output = output.replace("```", "")
        output = output.strip()

    try:
        result = json.loads(output)

        return {
            "factuality": float(result["factuality"]),
            "relevance": float(result["relevance"]),
            "hallucination": bool(result["hallucination"]),
            "explanation": result["explanation"]
        }

    except Exception as e:
        print("Failed to parse:")
        print(output)
        print("Error:", e)

        return {
            "factuality": None,
            "relevance": None,
            "hallucination": None,
            "explanation": None
        }

In [23]:
parsed = parse_judge_output(test_output)

parsed

{'factuality': 4.0,
 'relevance': 4.0,
 'hallucination': False,
 'explanation': 'The summary accurately captures key points from the source article without adding or omitting significant details.'}

In [24]:
qwen_eval_df = results_df.head(50).copy()

In [25]:
qwen_judge_results = []

for i, row in qwen_eval_df.iterrows():

    start_time = time.time()

    raw_output = evaluate_factuality(
        row["text"],
        row["prediction"]
    )

    parsed = parse_judge_output(raw_output)

    qwen_judge_results.append({
        "id": row["id"],
        "factuality": parsed["factuality"],
        "relevance": parsed["relevance"],
        "hallucination": parsed["hallucination"],
        "explanation": parsed["explanation"],
        "judge_time": time.time() - start_time
    })

    print(
        f"{i+1}/50 | "
        f"Factuality: {parsed['factuality']} | "
        f"Hallucination: {parsed['hallucination']}"
    )

1/50 | Factuality: 4.0 | Hallucination: False
2/50 | Factuality: 4.0 | Hallucination: False
Failed to parse:
?
   
2. Clarity and Conciseness:
   Is the summary clear and concise?

3. Comprehensiveness:
   Does the summary cover all key points from the original article?
Error: Expecting value: line 1 column 1 (char 0)
3/50 | Factuality: None | Hallucination: None
4/50 | Factuality: 4.0 | Hallucination: False
5/50 | Factuality: 4.0 | Hallucination: False
6/50 | Factuality: 4.0 | Hallucination: False
7/50 | Factuality: 4.0 | Hallucination: False
8/50 | Factuality: 4.0 | Hallucination: False
9/50 | Factuality: 4.0 | Hallucination: False
10/50 | Factuality: 4.0 | Hallucination: False
11/50 | Factuality: 4.0 | Hallucination: False
12/50 | Factuality: 4.0 | Hallucination: False
13/50 | Factuality: 4.0 | Hallucination: False
14/50 | Factuality: 4.0 | Hallucination: False
15/50 | Factuality: 4.0 | Hallucination: False
16/50 | Factuality: 4.0 | Hallucination: False
17/50 | Factuality: 4.0 | Hal

In [26]:
qwen_factuality_df = pd.DataFrame(qwen_judge_results)

qwen_factuality_df.head()

,id,factuality,relevance,hallucination,explanation,judge_time
0,0,4.0,4.0,False,The summary accurately captures key points fro...,3.000329
1,1,4.0,4.0,False,The summary accurately captures key points fro...,2.634454
2,2,NaN,NaN,None,NaN,3.368172
3,3,4.0,4.0,False,The summary accurately captures key points fro...,2.630862
4,4,4.0,4.0,False,The summary accurately captures key points fro...,2.664502


In [27]:
qwen_factuality_df.to_csv(
    "../results/qwen-factuality-results.csv",
    index=False,
    encoding="utf-8-sig"
)

In [28]:
print("Average Factuality:",
      qwen_factuality_df["factuality"].mean())

print("Average Relevance:",
      qwen_factuality_df["relevance"].mean())

print("Hallucination Rate:",
      qwen_factuality_df["hallucination"].mean())

print("Average Judge Time:",
      qwen_factuality_df["judge_time"].mean())

Average Factuality: 3.9591836734693877
Average Relevance: 4.0
Hallucination Rate: 0.0
Average Judge Time: 2.6473087310791015


In [29]:
mt5_results_df = pd.read_csv(
    "../results/mt5-arabic-results.csv"
)

print(mt5_results_df.shape)
mt5_results_df.head()

(200, 5)


,id,text,reference_summary,prediction,inference_time
0,0,حلت أمس الإثنين عناصر من الفرقة الوطنية للشرطة...,اختلالات مشروع “كُورْنِيشْ آسفي”.. الفرقة الوط...,قالت الجمعية الوطنية للشرطة القضائية المغربية ...,1.851679
1,1,علمت “كود” ان ندوة الرؤساء بمجلس المستشارين ال...,نايضة على بنشماس على لفلوس عاود ثاني . علاش كن...,أعلن مجلس النواب البارح ان عددا من اعضاء المجل...,1.096007
2,2,فالمغرب عارفين ومتأكدين باللي هاد القضية ديال ...,باش التقاليد كتغلب الجايحة. لمغاربة كيزورو وحد...,"هناك رجل اعمال بيضاوي لـ ""كود"" في المغرب الذي ...",0.948281
3,3,قالت مصادر محلية، أن لجنة تفتيش من وزارة الداخ...,لجنة تفتيشية من الداخلية طارت لجماعات فإقليم ا...,قالت مصادر محلية وطبية لبي بي سي إن جماعة امزو...,1.582940
4,4,قالت صفحة “اسرائيل تتكلم بالعربية”على “الفايسب...,اسرائيل لاعبة كبير وبالزربة مع المغرب. سفيرها ...,وصل مكتب الاتصال الإسرائيلي في المغرب إلى الرب...,0.877870


In [30]:
evaluated_ids = qwen_factuality_df["id"].tolist()

print(len(evaluated_ids))
print(evaluated_ids[:10])

50
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [31]:
mt5_eval_df = mt5_results_df[
    mt5_results_df["id"].isin(evaluated_ids)
].copy()

In [32]:
print("Number of samples:", len(mt5_eval_df))

Number of samples: 50


In [33]:
mt5_eval_df = (
    mt5_eval_df
    .set_index("id")
    .loc[evaluated_ids]
    .reset_index()
)

In [34]:
print(
    qwen_factuality_df["id"].head().tolist()
)

print(
    mt5_eval_df["id"].head().tolist()
)

[0, 1, 2, 3, 4]
[0, 1, 2, 3, 4]


In [35]:
mt5_judge_results = []

for i, row in mt5_eval_df.iterrows():

    start_time = time.time()

    raw_output = evaluate_factuality(
        row["text"],
        row["prediction"]
    )

    parsed = parse_judge_output(raw_output)

    mt5_judge_results.append({
        "id": row["id"],
        "factuality": parsed["factuality"],
        "relevance": parsed["relevance"],
        "hallucination": parsed["hallucination"],
        "explanation": parsed["explanation"],
        "judge_time": time.time() - start_time
    })

    print(
        f"{i+1}/50 | "
        f"Factuality: {parsed['factuality']} | "
        f"Hallucination: {parsed['hallucination']}"
    )

1/50 | Factuality: 4.0 | Hallucination: False
2/50 | Factuality: 4.0 | Hallucination: False
3/50 | Factuality: 4.0 | Hallucination: False
4/50 | Factuality: 4.0 | Hallucination: False
5/50 | Factuality: 5.0 | Hallucination: False
6/50 | Factuality: 4.0 | Hallucination: False
7/50 | Factuality: 4.0 | Hallucination: False
8/50 | Factuality: 4.0 | Hallucination: False
9/50 | Factuality: 4.0 | Hallucination: False
10/50 | Factuality: 4.0 | Hallucination: False
11/50 | Factuality: 4.0 | Hallucination: False
12/50 | Factuality: 4.0 | Hallucination: False
13/50 | Factuality: 2.0 | Hallucination: False
14/50 | Factuality: 4.0 | Hallucination: False
15/50 | Factuality: 4.0 | Hallucination: False
16/50 | Factuality: 4.0 | Hallucination: False
17/50 | Factuality: 4.0 | Hallucination: False
18/50 | Factuality: 4.0 | Hallucination: False
19/50 | Factuality: 4.0 | Hallucination: False
20/50 | Factuality: 4.0 | Hallucination: False
21/50 | Factuality: 4.0 | Hallucination: False
22/50 | Factuality: 4.

In [36]:
mt5_factuality_df = pd.DataFrame(
    mt5_judge_results
)

mt5_factuality_df.head()

,id,factuality,relevance,hallucination,explanation,judge_time
0,0,4.0,4.0,False,The summary accurately captures key points fro...,2.701394
1,1,4.0,4.0,False,The summary accurately reflects the key points...,2.647889
2,2,4.0,4.0,False,The summary accurately captures key points fro...,3.687975
3,3,4.0,4.0,False,The summary accurately captures key points fro...,2.596487
4,4,5.0,5.0,False,The summary accurately captures key points fro...,2.625259


In [37]:
mt5_factuality_df.to_csv(
    "../results/mt5-factuality-results.csv",
    index=False,
    encoding="utf-8-sig"
)

In [38]:
print(
    "Average Factuality:",
    mt5_factuality_df["factuality"].mean()
)

print(
    "Average Relevance:",
    mt5_factuality_df["relevance"].mean()
)

print(
    "Hallucination Rate:",
    mt5_factuality_df["hallucination"].mean()
)

print(
    "Average Judge Time:",
    mt5_factuality_df["judge_time"].mean()
)

Average Factuality: 3.94
Average Relevance: 4.02
Hallucination Rate: 0.0
Average Judge Time: 2.7377579689025877


In [ ]:
factuality_comparison = pd.DataFrame([
    {
        "model": "Qwen2.5-1.5B-Instruct",
        "avg_factuality": qwen_factuality_df["factuality"].mean(),
        "avg_relevance": qwen_factuality_df["relevance"].mean(),
        "hallucination_rate": qwen_factuality_df["hallucination"].mean(),
        "avg_judge_time": qwen_factuality_df["judge_time"].mean()
    },
    {
        "model": "mT5-small-Arabic-Summarization",
        "avg_factuality": mt5_factuality_df["factuality"].mean(),
        "avg_relevance": mt5_factuality_df["relevance"].mean(),
        "hallucination_rate": mt5_factuality_df["hallucination"].mean(),
        "avg_judge_time": mt5_factuality_df["judge_time"].mean()
    }
])

factuality_comparison

,model,avg_factuality,avg_relevance,hallucination_rate,avg_judge_time
0,Qwen2.5-1.5B-Instruct,3.959184,4.00,0.0,2.647309
1,mT5-small-Arabic-Summarization,3.940000,4.02,0.0,2.737758


In [8]:
benchmark = pd.DataFrame([
    {
        "Model": "Qwen2.5-1.5B-Instruct",

        # Quality
        "ROUGE-1": 0.0682749138,
        "ROUGE-2": 0.0015757576,
        "ROUGE-L": 0.0679491130,
        "BERTScore-F1": 0.6594252300,

        # Factuality
        "Factuality": qwen_fact["factuality"].mean(),
        "Relevance": qwen_fact["relevance"].mean(),
        "Hallucination Rate": qwen_fact["hallucination"].mean(),

        # Performance
        "Avg Latency (sec)": qwen_results["inference_time"].mean(),
        "Throughput (samples/min)": 60 / qwen_results["inference_time"].mean(),
        "Avg Output Length": qwen_results["prediction"].str.split().str.len().mean()
    },

    {
        "Model": "mT5-small-Arabic-Summarization",

        # Quality
        "ROUGE-1": 0.0270,
        "ROUGE-2": 0.0,
        "ROUGE-L": 0.0275,
        "BERTScore-F1": 0.6771834287,

        # Factuality
        "Factuality": mt5_fact["factuality"].mean(),
        "Relevance": mt5_fact["relevance"].mean(),
        "Hallucination Rate": mt5_fact["hallucination"].mean(),

        # Performance
        "Avg Latency (sec)": mt5_results["inference_time"].mean(),
        "Throughput (samples/min)": 60 / mt5_results["inference_time"].mean(),
        "Avg Output Length": mt5_results["prediction"].str.split().str.len().mean()
    }
])

benchmark

,Model,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore-F1,Factuality,Relevance,Hallucination Rate,Avg Latency (sec),Throughput (samples/min),Avg Output Length
0,Qwen2.5-1.5B-Instruct,0.068275,0.001576,0.067949,0.659425,3.959184,4.00,0.0,4.862519,12.339283,47.61
1,mT5-small-Arabic-Summarization,0.027000,0.000000,0.027500,0.677183,3.940000,4.02,0.0,1.164637,51.518176,18.11


In [9]:
benchmark.round(4)

,Model,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore-F1,Factuality,Relevance,Hallucination Rate,Avg Latency (sec),Throughput (samples/min),Avg Output Length
0,Qwen2.5-1.5B-Instruct,0.0683,0.0016,0.0679,0.6594,3.9592,4.00,0.0,4.8625,12.3393,47.61
1,mT5-small-Arabic-Summarization,0.0270,0.0000,0.0275,0.6772,3.9400,4.02,0.0,1.1646,51.5182,18.11


In [10]:
benchmark.to_csv(
    "../results/final_benchmark.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully!")

Saved successfully!
